# Step 4: Understanding Tools

        _Instructor Solution Notebook_  
        Estimated time: **60 minutes**  
        Difficulty: **Intermediate**

        ## Learning Objectives
        - [ ] Turn Python callables into tools with `@tool`.
- [ ] Pass tools into an agent constructor.
- [ ] Understand tool signatures and docstrings as part of the model contract.
- [ ] Inspect tool execution results through live runs.

        ## Prerequisites
        - Step 3 completed.
- Comfort reading type hints.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Tools let the model move beyond pure text generation and call real application logic. The key is to make that logic explicit, typed, and easy for the model to choose.

Expected output and validation notes:

Expected output snapshot:

- The agent uses `add_numbers`
- The final answer combines arithmetic with the tool-provided topic string


## Part 2: Core Implementation

### Define simple tools


In [ ]:
from agent_framework import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b

@tool
def get_current_focus() -> str:
    """Return the current course topic."""
    return "Tool anatomy in Microsoft Agent Framework"

agent = create_agent(
    name="ToolPrimer",
    instructions="You may use tools when they help answer factual questions.",
    tools=[add_numbers, get_current_focus],
)


## Part 2: Core Implementation

### Ask the model to choose a tool


In [ ]:
response = await agent.run("What is 17 plus 25, and what topic are we studying?")
print(response.text)
print_json(describe_response(response), title="Tool call response details")


## Part 3: Hands-On Exercises

Create a third tool that formats a repo path and ask the agent a question that requires it.

This mirrored notebook uses completed exercise code so instructors can demonstrate the target state.


In [ ]:
@tool
def format_repo_path(name: str) -> str:
    """Return an absolute path for a named top-level folder in the course repo."""
    root = resolve_notebook_root()
    return str(root / name)

path_agent = create_agent(
    name="PathAgent",
    instructions="Use tools to answer path and arithmetic questions accurately.",
    tools=[add_numbers, get_current_focus, format_repo_path],
)
response = await path_agent.run("What is 9 + 8 and where is the docs folder located?")
print(response.text)


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
@tool
def format_repo_path(name: str) -> str:
    """Return an absolute path for a named top-level folder in the course repo."""
    root = resolve_notebook_root()
    return str(root / name)

path_agent = create_agent(
    name="PathAgent",
    instructions="Use tools to answer path and arithmetic questions accurately.",
    tools=[add_numbers, get_current_focus, format_repo_path],
)
response = await path_agent.run("What is 9 + 8 and where is the docs folder located?")
print(response.text)


## Part 5: Best Practices & Tips

        - Prefer small, composable tools over one giant helper.
- Write docstrings as if the model will read them, because it effectively does.
- Keep return values clean and predictable for easy reuse in final responses.


## Summary & Key Takeaways

You converted ordinary Python functions into model-callable tools and saw them participate in a live answer.


## Additional Resources

        - `agent_framework FunctionTool and tool decorator docs`
- `helpers/llm.py`
